# Polynomial Regression — Model Complexity and Evaluation

**Course:** AIC 301 — Machine Learning  
**Instructor:** Dr. Kashif Sultan  
**Student:** Ahsan  
**Date:** October 14, 2026

---

## Problem Description

This notebook explores how **polynomial regression** can capture non-linear relationships that simple linear regression misses. We use the **Auto MPG dataset** (UCI Machine Learning Repository) to predict a car's fuel efficiency (`mpg`) from its engine power (`horsepower`).

The core question we investigate: *What polynomial degree best balances model complexity against generalization, and why is simply choosing the highest degree a bad strategy?*

### Dataset Summary
- **Source:** UCI Machine Learning Repository — Auto MPG (originally from Carnegie Mellon University)
- **Observations:** 398 car models (1970s–1980s era)
- **Target Variable:** `mpg` (miles per gallon) — continuous
- **Selected Input Feature:** `horsepower` — continuous
- **Why Regression?** The target `mpg` is a continuous numeric value, not a category, so this is a regression task.

In [ ]:
# ============================================================
# Step 1: Import Libraries
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: model building and evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Plot settings for cleaner visuals
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

---
## Step 2: Load and Explore the Dataset

We load the Auto MPG dataset and perform basic exploration to understand its structure, check for missing values, and examine the relationship between `horsepower` and `mpg`.

In [ ]:
# ============================================================
# Step 2a: Load the dataset
# ============================================================

df = pd.read_csv('auto_mpg.csv')

# Quick look at the first 5 rows
print('First 5 rows of the dataset:')
df.head()

In [ ]:
# ============================================================
# Step 2b: Dataset shape, data types, and missing values
# ============================================================

print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
print()

print('Column data types:')
print(df.dtypes)
print()

print('Missing values per column:')
print(df.isnull().sum())
print()

print('Basic statistics:')
df.describe()

In [ ]:
# ============================================================
# Step 2c: Visualize the relationship — Horsepower vs MPG
# ============================================================

plt.figure(figsize=(10, 6))
plt.scatter(df['horsepower'], df['mpg'], alpha=0.5, edgecolors='k', linewidth=0.5)
plt.xlabel('Horsepower', fontsize=12)
plt.ylabel('Miles Per Gallon (MPG)', fontsize=12)
plt.title('Horsepower vs MPG — Is the Relationship Linear?', fontsize=14)
plt.tight_layout()
plt.savefig('plots/01_horsepower_vs_mpg_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observation: The scatter plot shows a clear CURVED (non-linear) relationship.')
print('As horsepower increases, MPG decreases — but the drop is steep at first,')
print('then flattens out at higher horsepower values. A straight line would miss this curve.')

---
## Step 3: Data Cleaning and Train/Test Split

Before modeling, we handle missing values and split the data into training and test sets.

**Important:** We perform the split *before* fitting any transformations (like `PolynomialFeatures`). This prevents **data leakage** — the model should never see test data during training, not even indirectly through scaling or feature statistics.

In [ ]:
# ============================================================
# Step 3a: Handle missing values
# ============================================================

print(f'Rows before cleaning: {len(df)}')
print(f'Missing values in horsepower: {df["horsepower"].isnull().sum()}')

# Drop the 2 rows with missing horsepower values
# Reason: only 2 out of 398 rows (0.5%) — negligible data loss
# Alternative: we could fill with median, but dropping is simpler and more honest
df = df.dropna(subset=['horsepower'])

print(f'Rows after cleaning: {len(df)}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

In [ ]:
# ============================================================
# Step 3b: Select features and target
# ============================================================

# X = input feature (horsepower), y = target (mpg)
# We use double brackets [['horsepower']] to keep X as a 2D array (DataFrame)
# sklearn models require 2D input: (n_samples, n_features)
# Single brackets ['horsepower'] would give a 1D Series — sklearn would complain

X = df[['horsepower']]
y = df['mpg']

print(f'Feature matrix X shape: {X.shape}  — {X.shape[0]} samples, {X.shape[1]} feature')
print(f'Target vector y shape:  {y.shape}  — {y.shape[0]} values')

In [ ]:
# ============================================================
# Step 3c: Train/Test Split
# ============================================================

# 80% training, 20% testing
# random_state=42 ensures reproducibility — same split every time we run
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print()
print('The same train/test split will be used for ALL models (linear and polynomial)')
print('to ensure a fair comparison.')

---
## Step 4: Baseline — Linear Regression

We start with a simple **Linear Regression** model as our baseline. This fits a straight line through the data:

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{horsepower}$$

The baseline tells us how well a straight line can explain the horsepower–mpg relationship. Any polynomial model must beat this to justify its added complexity.

In [ ]:
# ============================================================
# Helper function: Evaluate any regression model
# ============================================================
# We define this once and reuse it for every model (linear, poly deg 2, 3, 4, ...)
# This avoids repeating the same metric-calculation code multiple times.

def evaluate_model(model, X_train, X_test, y_train, y_test):
    """Calculate R², RMSE, and MAE on both training and test sets."""
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics on training data
    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_mae  = mean_absolute_error(y_train, y_train_pred)
    
    # Metrics on test data
    test_r2   = r2_score(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae  = mean_absolute_error(y_test, y_test_pred)
    
    return {
        'Train R²': train_r2, 'Test R²': test_r2,
        'Train RMSE': train_rmse, 'Test RMSE': test_rmse,
        'Train MAE': train_mae, 'Test MAE': test_mae
    }

In [ ]:
# ============================================================
# Step 4: Train the Linear Regression baseline
# ============================================================

# Create and train the model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Print the learned equation: mpg = intercept + slope * horsepower
print('Learned equation:')
print(f'  mpg = {linear_model.intercept_:.2f} + ({linear_model.coef_[0]:.4f}) × horsepower')
print()
print(f'Interpretation: For every 1-unit increase in horsepower,')
print(f'the model predicts MPG decreases by {abs(linear_model.coef_[0]):.4f} miles per gallon.')
print(f'The negative coefficient confirms the inverse relationship we saw in the scatter plot.')

In [ ]:
# ============================================================
# Step 4b: Evaluate the baseline
# ============================================================

linear_results = evaluate_model(linear_model, X_train, X_test, y_train, y_test)

print('=== Linear Regression (Baseline) Performance ===')
print(f'  Training R²:   {linear_results["Train R²"]:.4f}')
print(f'  Test R²:       {linear_results["Test R²"]:.4f}')
print(f'  Training RMSE: {linear_results["Train RMSE"]:.4f} mpg')
print(f'  Test RMSE:     {linear_results["Test RMSE"]:.4f} mpg')
print(f'  Training MAE:  {linear_results["Train MAE"]:.4f} mpg')
print(f'  Test MAE:      {linear_results["Test MAE"]:.4f} mpg')
print()
print('Interpretation:')
print('An R² around 0.60 means the straight line explains about 60% of the variance in mpg.')
print('The remaining 40% is unexplained — likely because the true relationship is curved.')
print('This motivates trying polynomial regression to capture that non-linearity.')

In [ ]:
# ============================================================
# Step 4c: Visualize the linear fit
# ============================================================

# Create a smooth range of horsepower values for plotting the fitted line
X_range = np.linspace(X['horsepower'].min(), X['horsepower'].max(), 300).reshape(-1, 1)
y_linear_pred = linear_model.predict(X_range)

plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, alpha=0.5, edgecolors='k', linewidth=0.5, label='Test data')
plt.plot(X_range, y_linear_pred, color='red', linewidth=2, label='Linear fit (degree 1)')
plt.xlabel('Horsepower', fontsize=12)
plt.ylabel('Miles Per Gallon (MPG)', fontsize=12)
plt.title('Linear Regression Baseline — Straight Line Fit', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('plots/02_linear_regression_fit.png', dpi=150, bbox_inches='tight')
plt.show()

print('The red line clearly misses the curve in the data.')
print('It over-predicts MPG at very low and very high horsepower,')
print('and under-predicts in the middle range. This is a sign of UNDERFITTING.')